In [1]:
pip install torch transformers datasets scikit-learn accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Importing the necessary libraries
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup


In [3]:
# Loading the dataset
data = pd.read_csv("dataset/train.csv")

In [ ]:
# Convert one-hot winner columns into a single 'preferred' column
def get_preferred(row):
    if row["winner_model_a"] == 1:
        return 0  # A wins
    elif row["winner_model_b"] == 1:
        return 1  # B wins
    else:
        return 2  # Tie


data["preferred"] = data.apply(get_preferred, axis=1)


# Should show counts for 0, 1, and 2
print(data["preferred"].value_counts())


preferred
0    19803
1    19349
2    17492
Name: count, dtype: int64


In [5]:
data.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie,preferred
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1.0,0.0,0.0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0.0,1.0,0.0,1
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0.0,0.0,1.0,2
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1.0,0.0,0.0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0.0,1.0,0.0,1


In [ ]:
# Splitting data into train and validation


train_data_dummy, temp_data = train_test_split(
    data,
    test_size=0.1,
    random_state=58,
    stratify=data["preferred"],
)  # keeps class balance (A/B/Tie) equal in all splits

# Splitting the data so that we only use half and not the whole thing since we have a very large dataset
train_data, train_data_not_to_use = train_test_split(
    train_data_dummy,
    test_size=0.7,
    random_state=58,
    stratify=train_data_dummy["preferred"],
)


# Second split: split the 20% into 10% val and 10% test
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=58, stratify=temp_data["preferred"]
)


# Reset indexes
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

print(f"Train size:      {len(train_data)}")
print(f"Validation size: {len(val_data)}")
print(f"Test size:       {len(test_data)}")

# Check class balance is maintained
print("\nTrain distribution:")
print(train_data["preferred"].value_counts())
print("\nVal distribution:")
print(val_data["preferred"].value_counts())
print("\nTest distribution:")
print(test_data["preferred"].value_counts())


Train size:      45315
Validation size: 5664
Test size:       5665

Train distribution:
preferred
0    15842
1    15479
2    13994
Name: count, dtype: int64

Val distribution:
preferred
0    1980
1    1935
2    1749
Name: count, dtype: int64

Test distribution:
preferred
0    1981
1    1935
2    1749
Name: count, dtype: int64


In [ ]:
class RewardModel(nn.Module):
    def __init__(self, model_name="roberta-base", num_classes=3):  # 3 classes now
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size  # 768

        # We encode A and B separately, then concatenate before classifying
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 512),  # *2 because we concat A and B
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes),  # outputs 3 scores
        )

    def encode_response(self, input_ids, attention_mask):
        output = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return output.last_hidden_state[:, 0, :]  # CLS token

    def forward(self, input_ids_a, mask_a, input_ids_b, mask_b):
        emb_a = self.encode_response(input_ids_a, mask_a)
        emb_b = self.encode_response(input_ids_b, mask_b)

        # Concatenate both embeddings so model sees A and B together
        combined = torch.cat([emb_a, emb_b], dim=1)  # shape: (batch, 768*2)
        logits = self.classifier(combined)  # shape: (batch, 3)
        return logits


In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
MAX_LEN = 256


In [ ]:
class PreferenceDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe[
            ["prompt", "response_a", "response_b", "preferred"]
        ].to_dict(orient="records")

    def encode(self, prompt, response):
        text = f"[Prompt]: {prompt} [Response]: {response}"
        return tokenizer(
            text,
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        enc_a = self.encode(str(item["prompt"]), str(item["response_a"]))
        enc_b = self.encode(str(item["prompt"]), str(item["response_b"]))

        return {
            "input_ids_a": enc_a["input_ids"].squeeze(),
            "attention_mask_a": enc_a["attention_mask"].squeeze(),
            "input_ids_b": enc_b["input_ids"].squeeze(),
            "attention_mask_b": enc_b["attention_mask"].squeeze(),
            "label": torch.tensor(
                item["preferred"], dtype=torch.long
            ),  
        }


In [ ]:
EPOCHS = 1
BATCH_SIZE = 2
LR = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 3

# Using the split datasets, not the full data
train_dataset = PreferenceDataset(train_data)
val_dataset = PreferenceDataset(val_data)
test_dataset = PreferenceDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = RewardModel(num_classes=NUM_CLASSES).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)


def evaluate(model, dataloader):
    model.eval()
    total_loss = 0
    correct = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids_a = batch["input_ids_a"].to(DEVICE)
            mask_a = batch["attention_mask_a"].to(DEVICE)
            input_ids_b = batch["input_ids_b"].to(DEVICE)
            mask_b = batch["attention_mask_b"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            logits = model(input_ids_a, mask_a, input_ids_b, mask_b)
            loss = F.cross_entropy(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / len(dataloader.dataset)
    return avg_loss, accuracy


# --- Training Loop ---
best_val_accuracy = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0

    for batch in train_loader:
        input_ids_a = batch["input_ids_a"].to(DEVICE)
        mask_a = batch["attention_mask_a"].to(DEVICE)
        input_ids_b = batch["input_ids_b"].to(DEVICE)
        mask_b = batch["attention_mask_b"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        logits = model(input_ids_a, mask_a, input_ids_b, mask_b)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()

    train_loss = total_loss / len(train_loader)
    train_acc = correct / len(train_dataset)

    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch + 1}")
    print(f"  Train -> Loss: {train_loss:.4f} | Accuracy: {train_acc:.4f}")
    print(f"  Val   -> Loss: {val_loss:.4f} | Accuracy: {val_acc:.4f}")

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save(model.state_dict(), "best_model.pt")
        print(f" Best model saved (val acc: {val_acc:.4f})")


In [ ]:
# Final Evaluation stage

# Load the best saved model
model.load_state_dict(torch.load("best_model.pt"))

test_loss, test_acc = evaluate(model, test_loader)
print(f"\nFinal Test Results")
print(f"  Loss:     {test_loss:.4f}")
print(f"  Accuracy: {test_acc:.4f}")
